# iKwath process model

Physics and signal models for the iKwath decoction cycle (SIH26048). Everything here is a **calculation from design parameters**, not a measurement from a built machine. Its job is to check whether the design targets in the README are consistent with each other before hardware is built.

Contents
1. Boiling point of water versus chamber pressure (vacuum boiling)
2. Energy budget: how much heater power a given cycle time needs
3. Time-domain simulation of the heat-up and reduction phases
4. Load-cell endpoint detection with sensor noise
5. Pod tag payload encoding and checksum (mirrors `ino/ikwath_firmware/profile.h`)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Constants (approximate values for water near 80-100 C) ----
CP_WATER = 4.19        # kJ/(kg K)
H_FG     = 2300.0      # kJ/kg latent heat of vaporisation near 85 C
KPA_PER_MMHG = 0.133322

# ---- Design parameters from the README ----
WATER_G       = 400.0  # water charge
EVAP_TARGET_G = 300.0  # evaporated mass for a 4:1 reduction
T_START_C     = 25.0   # tap-water temperature
T_SET_C       = 86.0   # boil setpoint (inside 85-90 C band)
HEATER_W_BOM  = 50.0   # 12 V silicone heater in the current bill of materials
HEAT_EFF      = 0.80   # fraction of heater power that reaches the water (assumption)

## 1. Vacuum boiling point

Antoine equation for water (valid roughly 1 to 100 C): `log10(P_mmHg) = A - B / (C + T)`.

In [ ]:
A, B, C = 8.07131, 1730.63, 233.426

def p_sat_kpa(t_c):
    return 10 ** (A - B / (C + np.asarray(t_c, dtype=float))) * KPA_PER_MMHG

def boiling_point_c(p_kpa):
    p_mmhg = np.asarray(p_kpa, dtype=float) / KPA_PER_MMHG
    return B / (A - np.log10(p_mmhg)) - C

pressures = np.linspace(20, 101.3, 200)
bp = boiling_point_c(pressures)

for p in (101.3, 60, 55, 50):
    print(f"chamber {p:6.1f} kPa  ->  water boils at {boiling_point_c(p):5.1f} C")

plt.figure(figsize=(6, 3.6))
plt.plot(pressures, bp)
plt.axhspan(85, 90, alpha=0.2, label="85-90 C target band")
plt.axvspan(50, 60, alpha=0.15, label="0.50-0.60 atm-ish window")
plt.xlabel("chamber pressure (kPa absolute)")
plt.ylabel("boiling point (C)")
plt.title("Vacuum boiling point of water")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Reading the result: the design window (about 50 to 60 kPa) puts the boiling point at roughly 81 to 86 C. That stays under the 90 C ceiling that matters for the herb, but the lower part of the window boils below 85 C, so the 85 C figure is a ceiling-side target rather than something vacuum boiling will hold. Decide whether 85 C is meant as a floor; if it is not, the design is consistent. The heater setpoint in the firmware sits just above the highest boiling point, so during the boil the heater runs at full duty, which is what the energy budget below needs.

## 2. Energy budget

Total energy = sensible heat to warm the water + latent heat to evaporate the target mass. Latent heat dominates, and vacuum does not reduce it.

In [ ]:
def energy_kj(water_g=WATER_G, evap_g=EVAP_TARGET_G, t0=T_START_C, t1=T_SET_C):
    sensible = water_g / 1000 * CP_WATER * (t1 - t0)
    latent   = evap_g / 1000 * H_FG
    return sensible, latent

def cycle_minutes(power_w, eff=HEAT_EFF, **kw):
    s, l = energy_kj(**kw)
    return (s + l) * 1000 / (power_w * eff) / 60

def power_needed_w(minutes, eff=HEAT_EFF, **kw):
    s, l = energy_kj(**kw)
    return (s + l) * 1000 / (minutes * 60 * eff)

sens, lat = energy_kj()
print(f"sensible heat : {sens:7.1f} kJ")
print(f"latent heat   : {lat:7.1f} kJ   ({lat / (sens + lat):.0%} of the total)")
print()
print(f"BOM heater ({HEATER_W_BOM:.0f} W, {HEAT_EFF:.0%} efficient): {cycle_minutes(HEATER_W_BOM):7.0f} minutes for the full cycle")
for target in (12, 20, 30, 45):
    print(f"to finish in {target:2d} min you need about {power_needed_w(target):6.0f} W into the heater")

In [ ]:
powers = np.linspace(50, 3000, 300)
plt.figure(figsize=(6, 3.6))
plt.plot(powers, [cycle_minutes(p) for p in powers], label="full 4:1 reduction (300 g)")
plt.plot(powers, [cycle_minutes(p, evap_g=150) for p in powers], label="half reduction (150 g)")
plt.axhline(12, ls="--", color="k", alpha=0.5, label="12 min design target")
plt.axvline(HEATER_W_BOM, ls=":", color="r", label="BOM heater (50 W)")
plt.yscale("log")
plt.xlabel("heater power (W)")
plt.ylabel("cycle time (minutes, log scale)")
plt.title("Cycle time versus heater power")
plt.legend(fontsize=8)
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

**Finding.** With the 50 W heater in the current bill of materials the full cycle takes hours, not 9 to 12 minutes. A 12 minute cycle needs on the order of 1.5 kW into the water. That points to a mains heating element (with matching relay, wiring and thermal-cutoff ratings), a smaller reduction, or a longer cycle target. This is an open design decision and should be settled before the PCB power stage and the bill of materials are frozen.

In [ ]:
def simulate(power_w, eff=HEAT_EFF, dt=1.0, t_max_s=6 * 3600, noise_g=0.0, seed=0, overrun_g=0.0):
    rng = np.random.default_rng(seed)
    t, temp, mass = 0.0, T_START_C, WATER_G
    evap = 0.0
    rows = []
    while t < t_max_s and evap < EVAP_TARGET_G + overrun_g:
        q_kj = power_w * eff * dt / 1000
        if temp < T_SET_C:
            temp += q_kj / ((mass / 1000) * CP_WATER)      # heating phase
            temp = min(temp, T_SET_C)
        else:
            d_evap = q_kj / H_FG * 1000                    # grams evaporated this step
            evap += d_evap
            mass -= d_evap
        measured = mass + rng.normal(0, noise_g) if noise_g else mass
        rows.append((t, temp, mass, evap, measured))
        t += dt
    return np.array(rows)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for w, label in ((HEATER_W_BOM, "50 W"), (1200, "1.2 kW"), (1800, "1.8 kW")):
    r = simulate(w, dt=1.0 if w >= 500 else 10.0)
    ax[0].plot(r[:, 0] / 60, r[:, 1], label=label)
    ax[1].plot(r[:, 0] / 60, r[:, 3], label=label)
    print(f"{label:>7}: reached {EVAP_TARGET_G:.0f} g evaporated after {r[-1, 0] / 60:6.1f} min")
ax[0].axhspan(85, 90, alpha=0.2)
ax[0].set_xscale("log"); ax[1].set_xscale("log")
ax[0].set_xlabel("time (min, log)"); ax[0].set_ylabel("liquid temperature (C)")
ax[1].set_xlabel("time (min, log)"); ax[1].set_ylabel("evaporated mass (g)")
ax[1].axhline(EVAP_TARGET_G, ls="--", color="k", alpha=0.5)
ax[0].legend(); ax[0].grid(alpha=0.3); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Endpoint detection with load-cell noise

The firmware ends the boil when the filtered mass loss reaches the target. This block checks how far the stopping point can drift from the true value for a given noise level, using a short moving average.

In [ ]:
def endpoint_error(noise_g, window=5, power_w=1500, trials=200):
    errs, misses = [], 0
    kern = np.ones(window) / window
    for k in range(trials):
        r = simulate(power_w, noise_g=noise_g, seed=k, overrun_g=30.0)   # keep logging past the target
        meas = r[:, 4]
        m0 = meas[:10].mean()                        # baseline from the first ten samples
        avg = np.convolve(meas, kern, mode="valid")
        loss = m0 - avg
        hits = np.nonzero(loss >= EVAP_TARGET_G - 2.0)[0]
        if len(hits) == 0:
            misses += 1
            continue
        idx = min(hits[0] + window - 1, len(r) - 1)  # the sample the average is aligned to
        errs.append(r[idx, 3] - EVAP_TARGET_G)       # true evaporated mass minus the target
    return np.mean(errs), np.std(errs), misses

print("noise (g rms)   mean error (g)   spread (g)   missed")
for n in (0.2, 0.5, 1.0, 2.0, 4.0):
    mu, sd, miss = endpoint_error(n)
    print(f"{n:11.1f}   {mu:13.2f}   {sd:9.2f}   {miss:6d}")

The design tolerance is about 2 g. The table shows the noise level the load cell, mounting and hose forces must stay under to meet it. Measure the real noise floor on the bench, including with the vacuum pump running, and compare.

## 5. Pod tag payload

Encode and decode the 16-byte profile stored on the pod's NTAG213 tag. Byte layout and validation rules mirror `ino/ikwath_firmware/profile.h`.

In [ ]:
BAND_LOW, BAND_HIGH = 85.0, 90.0

def encode_profile(ratio_class, presoak_s, evap_g, setpoint_c, tds_min_ppm, water_ml, dose_ml):
    b = bytearray(16)
    b[0], b[1], b[2] = ord('I'), ord('K'), 1
    b[3] = ratio_class
    b[4:6] = int(presoak_s).to_bytes(2, "little")
    b[6:8] = int(evap_g).to_bytes(2, "little")
    b[8] = int(setpoint_c)
    b[9] = int(tds_min_ppm // 10)
    b[10:12] = int(water_ml).to_bytes(2, "little")
    b[12] = int(dose_ml)
    x = 0
    for v in b[:15]:
        x ^= v
    b[15] = x
    return bytes(b)

def decode_profile(b):
    if len(b) != 16 or b[0] != ord('I') or b[1] != ord('K') or b[2] != 1:
        return None
    x = 0
    for v in b[:15]:
        x ^= v
    if x != b[15]:
        return None
    soak = int.from_bytes(b[4:6], "little")
    evap = int.from_bytes(b[6:8], "little")
    water = int.from_bytes(b[10:12], "little")
    setpoint = b[8]
    if not (BAND_LOW <= setpoint <= BAND_HIGH - 1.0):
        return None
    if not (100 <= water <= 600) or evap == 0 or evap >= water or soak > 900:
        return None
    return dict(ratio_class=b[3], presoak_s=soak, evap_g=evap, setpoint_c=setpoint,
                tds_min_ppm=b[9] * 10, water_ml=water, dose_ml=b[12] or 100)

tag = encode_profile(8, 180, 300, 86, 0, 400, 100)
print("payload hex :", tag.hex(" "))
print("decoded     :", decode_profile(tag))

corrupt = bytearray(tag); corrupt[6] ^= 0x01
print("corrupted   :", decode_profile(bytes(corrupt)), "(rejected by checksum)")
print("hot setpoint:", decode_profile(encode_profile(8, 180, 300, 96, 0, 400, 100)), "(rejected by envelope check)")

# Herb-class water ratios from the formulary and the resulting profiles for a 25 g pod
for name, ratio in (("soft", 16), ("medium", 8), ("hard", 4)):
    print(f"{name:>6}: {ratio:2d}x water -> {25 * ratio:4d} mL for a 25 g charge")

## Notes and limits

* All properties are textbook approximations. Heater efficiency, heat losses to the chamber wall and lid, and the effect of vacuum on evaporation kinetics are not modelled in detail.
* The simulation holds temperature at the setpoint once reached, which is optimistic; a real PID will overshoot and settle.
* Nothing here has been validated against measurements. Replace the assumptions with bench data as soon as a prototype exists.